# 11 — Final predictions and attributions

**Objective.** Verify both protocol locks, log final-test access, execute immutable chunked model/refit tasks on 2010, and merge long-form predictions/attributions only after all chunks complete.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Permitted block: untouched 2010 final test. Every access is appended to a hash-chained log.

In [ ]:
import os
CHUNK_INDEX = int(os.environ.get("CRUX_CHUNK_INDEX", "0"))
MERGE_ONLY = os.environ.get("CRUX_MERGE_ONLY", "0") == "1"
STAGE_SUFFIX = None if MERGE_ONLY else f"chunk_{CHUNK_INDEX:03d}"

# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("11", suffix=STAGE_SUFFIX)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import joblib, json
import pandas as pd
from cruxvc.explanations import load_feature_groups
from cruxvc.io import read_json, read_table
from cruxvc.manifest import StageRecorder, append_test_access_log, signing_key_from_environment, verify_protocol_lock
from cruxvc.workflow import merge_chunk_outputs, run_final_task_chunk

signing_key = signing_key_from_environment()
require_hmac = bool(CFG["execution"]["require_hmac_for_final_test"])
verify_protocol_lock(P.locks / "phase0_lock.json", signing_key=signing_key, require_hmac=require_hmac)
design_lock = verify_protocol_lock(P.locks / "design_lock.json", signing_key=signing_key, require_hmac=require_hmac)
if not design_lock.get("final_grade"):
    raise RuntimeError("Design lock is not final-grade. Re-run development and Notebook 10 under CRUX_PROFILE=full.")
task_registry = read_table(P.protocol / "final_task_registry.parquet")
chunk_dir = P.results / "stage11_chunks"
chunk_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
if MERGE_ONLY:
    expected_chunks = sorted(task_registry["chunk_index"].unique())
    missing_manifests = [
        index for index in expected_chunks
        if not (P.manifests / f"11_final_predictions_and_attributions_chunk_{int(index):03d}.json").exists()
    ]
    if missing_manifests:
        raise RuntimeError(f"Cannot merge; missing completed chunk manifests: {missing_manifests}")
    final_predictions = P.predictions / "final_predictions.parquet"
    final_attributions = P.attributions / "final_attributions_long.parquet"
    merge_chunk_outputs(chunk_dir, final_predictions, final_attributions)
    append_test_access_log(P, stage_id="11", purpose="merge immutable final-test task chunks", resources=[final_predictions, final_attributions])
    CTX.recorder.complete([final_predictions, final_attributions], extra={"merged_chunks": expected_chunks})
    print(f"Merged {len(expected_chunks)} chunks into canonical Stage-11 outputs.")

In [ ]:
if not MERGE_ONLY:
    tasks = task_registry[task_registry["chunk_index"].eq(CHUNK_INDEX)].copy()
    if tasks.empty:
        raise RuntimeError(f"No final tasks assigned to chunk {CHUNK_INDEX}")
    append_test_access_log(P, stage_id="11", purpose=f"execute locked final-test chunk {CHUNK_INDEX}", resources=[P.processed / "cohort_labels.parquet", P.processed / "features_strict.parquet"])

    features = read_table(P.processed / "features_strict.parquet")
    cohort = read_table(P.processed / "cohort_labels.parquet")
    splits = read_table(P.protocol / "split_ids.parquet")
    data = features.merge(cohort, on=["case_id", "company_permalink", "t0"]).merge(splits[["case_id", "time_block"]], on="case_id")
    training = data[data["time_block"].astype(str).eq("development")].copy()
    probability_calibration = data[data["time_block"].astype(str).eq("probability_calibration")].copy()
    test = data[data["time_block"].astype(str).eq("final_test")].copy()
    audit_registry = pd.read_csv(P.protocol / "local_audit_case_ids.csv")
    audit_design = audit_registry.drop(columns=["landmark_round_type"], errors="ignore")
    audit_cases = test[test["case_id"].isin(audit_registry["case_id"])].merge(
        audit_design,
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    background_ids = pd.read_csv(P.protocol / "explanation_background_ids.csv")
    backgrounds = {
        background_id: training[training["case_id"].isin(group.sort_values("order")["case_id"])].copy()
        for background_id, group in background_ids.groupby("background_id")
    }
    seeds = pd.read_csv(P.protocol / "seed_registry.csv")
    approximation_seeds = seeds[seeds["purpose"].eq("approximation")].sort_values("index")["seed"].head(int(PROFILE["approximation_seeds"])).astype(int).tolist()
    feature_columns = [column for column in features.columns if column not in {"case_id", "company_permalink", "t0"}]
    groups = load_feature_groups(P.config / "feature_groups.yaml")
    prediction_path, attribution_path = run_final_task_chunk(
        tasks, training, probability_calibration, test, audit_cases, backgrounds, groups, feature_columns,
        approximation_seeds, int(PROFILE["permutation_orderings"]), chunk_dir,
    )
    CTX.recorder.complete([prediction_path, attribution_path], extra={"chunk_index": CHUNK_INDEX, "task_count": len(tasks)})
    print(f"Chunk {CHUNK_INDEX} completed with {len(tasks)} locked tasks. Set CRUX_MERGE_ONLY=1 only after every planned chunk is complete.")

In [ ]:
if not MERGE_ONLY and task_registry["chunk_index"].nunique() == 1:
    final_predictions = P.predictions / "final_predictions.parquet"
    final_attributions = P.attributions / "final_attributions_long.parquet"
    merge_chunk_outputs(chunk_dir, final_predictions, final_attributions)
    canonical = StageRecorder(P, "11", inputs=[P.protocol / "final_task_registry.parquet"], permitted_blocks=["final_test"])
    canonical.complete([final_predictions, final_attributions], extra={"merged_chunks": [0]})
    print("Single-chunk run merged automatically.")